In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
from tqdm.notebook import tqdm

tfd = tfp.distributions
tfb = tfp.bijectors
from bakeoff.TensorFlow_Prob.run_tfp import make_conditioned_lp

In [2]:
from modulars.utils import load_config, load_best_values, exp_lognormal_moments
from modulars.distributions import gaussian_1d
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d
from modulars.tfp_rr_test import tfp_run_restart_1d


In [3]:
config_file = '../invgamma_config.json'
config = load_config(config_file)
mu_like = config['mu_like']
sigma_like = config['sigma_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']
grad_samps = config['grad_samps']
max_iters = 100_000#config['max_iters']


DATA = True
if DATA:
  data = gaussian_1d(n_samples, mu_like, sigma_like=sigma_like).astype(np.float32)
else:
  data = []


conditioned_log_prob = make_conditioned_lp(
  prior_dist = tfd.InverseGamma(alpha_prior, beta_prior),
  likelihood_dist = lambda v: tfd.Normal(mu_like, tf.sqrt(v)),
  x = data
)
print("Prior is InverseGamma with alpha =", alpha_prior, "and beta =", beta_prior)
print("Likelihood is Normal with mean =", mu_like, "and std =", sigma_like)
print("data is", data)

Prior is InverseGamma with alpha = 4.0 and beta = 3.0
Likelihood is Normal with mean = 3.0 and std = 0.5
data is [3.4419465 3.0979326 3.1787682 1.828369  2.4575837 3.279848  3.4697347
 2.5107596 3.2515485 3.2032073]


In [4]:
results = []
bij = tfb.Exp()
for seed in tqdm(range(100)):
    result = tfp_run_restart_1d(
        seed, conditioned_log_prob, bij)
    results.append(result)

  0%|          | 0/100 [00:00<?, ?it/s]

2026-03-16 20:59:44.864955: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. fit_surrogate_posterior/sanitize_seed/seed
I0000 00:00:1773709185.096001 2081108 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
%matplotlib inline
from modulars import apply_traj_transform, save_rr_tracking_csv
TRACKING_CSV = "processed_tracking/rr_tfp_invgam_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results, transform_fn=exp_lognormal_moments,
                     n_samples=10_000, seed=1, NOTEBOOK=True)
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
)


In [ ]:
%matplotlib inline
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_tfp_invgam_tracking.csv"

single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)
N, T = single_means.shape

# If we have "best" from config, these should be on theta-scale (0,1)
best_mu, best_std = load_best_values(
    config=config, transform=exp_lognormal_moments,
    n_samples=50_000, seed=1)


plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r'$\sigma^2$')
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r'$\sigma^2$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r'$\sigma^2$', 100)
